## Einops tutorial part 2, deep learning

Based on https://einops.rocks/2-einops-for-deep-learning/

In [1]:
from einops import rearrange, reduce
import numpy as np

In [2]:
x = np.random.RandomState(42).normal(size=[10, 32, 100, 200])

In [3]:
import torch

In [4]:
x = torch.from_numpy(x)
x.require_grad = True

In [5]:
type(x), x.shape

(torch.Tensor, torch.Size([10, 32, 100, 200]))

In [6]:
# converting bchw to bhwc format is a common operation in OpenCV
# einops operation support deep learning frameworks

y = rearrange(x, "b c h w -> b h w c")
y.shape

torch.Size([10, 100, 200, 32])

## Backpropagation

In [7]:
# You can backpropagate through einops operations

y0 = x
y1 = reduce(y0, "b c h w -> b c", "max")
y2 = rearrange(y1, "b c -> c b")
y3 = reduce(y2, "c b -> ", "sum")

y3.backward() # This example doesn't work
print(reduce(x.grad, "b c h w -> ", "sum"))

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

## Meet `einops.asnumpy`

Converts tensors to numpy (and pulls from GPU if necessary)

In [9]:
from einops import asnumpy

print(type(y3))
y3_numpy = asnumpy(y3)
print(type(y3_numpy))

<class 'torch.Tensor'>
<class 'numpy.ndarray'>


## Common building blocks for deep learning

In [12]:
# Flattening is a common operation, frequently appears at the boundary between convolutional and fully-connected layers.

y = rearrange(x, "b c h w -> b (c h w)")
print(y.shape)

torch.Size([10, 640000])


In [13]:
# space-to-depth

y = rearrange(x, "b c (h h1) (w w1) -> b (h1 w1 c) h w", h1=2, w1=2)
print(y.shape)

torch.Size([10, 128, 50, 100])


In [14]:
# depth-to-space (reverse of the previous)
y = rearrange(x, "b (h1 w1 c) h w -> b c (h h1) (w w1)", h1=2, w1=2)
print(y.shape)

torch.Size([10, 8, 200, 400])


## Reductions

In [15]:
# Simple global average pooling

y = reduce(x, "b c h w -> b c", "mean")
print(y.shape)

torch.Size([10, 32])


In [18]:
# Max pooling with 2x2 kernel

y = reduce(x, "b c (h h1) (w w1) -> b c h w", "max", h1=2, w1=2)
print(y.shape)

torch.Size([10, 32, 50, 100])


In [21]:
# You can skip names for reduced axes

y = reduce(x, "b c (h 2) (w 2) -> b c h w", reduction="max")
print(y.shape)

torch.Size([10, 32, 50, 100])


## 1d, 2d and 3d pooling are defined in a similar way

In [26]:
# for sequential, 1d models, you'll probably want pooling over time

y = reduce(x, "(t 2) b c d -> t b c d", reduction="max")
print(y.shape)

torch.Size([5, 32, 100, 200])


In [27]:
# for volumetric models, all 3 dimensions are pooled

y = reduce(x, "b (x 2) (y 2) (z 2) -> b x y z", reduction="max")
print(y.shape)

torch.Size([10, 16, 50, 100])


## Squeeze and unsqueeze (expand_dims)

In [29]:
# models typically work only with batches,
# so top predict a single image ...
image = rearrange(x[0, :3], "c h w -> h w c")
# ... create a dummy 1-element axis ...
y = rearrange(image, "h w c -> () c h w")
# ... imagine you predicted this with a convolutional network for classification,
# we'll just flatten axes ...
predictions = rearrange(y, "b c h w -> b (c h w)")
# ... finally, decompose (remove) dummy axis
predictions = rearrange(predictions, "() classes -> classes")
print(predictions.shape)

torch.Size([60000])


## keepdmis-like behaviour for reductions

- empty decompositions `()` provides dimensions of length 1, which are broadcastable
- alternatively, you can use `1` to introduce new axis, that's a synonym to `()`

In [33]:
# per-channel mean-normalization for each image

y = x - reduce(x, "b c h w -> b c 1 1", "mean")
print(y.shape)

torch.Size([10, 32, 100, 200])


In [34]:
# per-channel mean-normalization for whole batch:

y = x - reduce(y, "b c h w -> 1 c 1 1", "mean")
print(y.shape)

torch.Size([10, 32, 100, 200])
